In [ ]:
from transformers import PreTrainedTokenizerFast
from datasets import Dataset, load_dataset
from tqdm import tqdm
import os

DATA_DIR = "/home/wyf/orcd/pool/reverse-llm/data"
TOKENIZER_DIR = "/home/wyf/orcd/pool/reverse-llm/tokenizers"

USER_ROLE_NAME = "user"[::-1]
ASSISTANT_ROLE_NAME = "assistant"[::-1]

dataset_name = "ultrachat"
context_length = 1024

# Load tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(f"{TOKENIZER_DIR}/fineweb_bpe_200k")
tokenizer.add_special_tokens({"additional_special_tokens": ["<im_start>", "<im_end>"]})

tokenizer.chat_template = """{% for message in messages -%}
<im_start>{{ message['role'] }}
{{ message['content'] }}<im_end>
{%- endfor -%}
{% if add_generation_prompt and messages[-1]['role'] != 'assistant' -%}
<im_start>assistant
{%- endif %}"""

# Load dataset
raw_dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split=["train_sft", "test_sft"])

def process_ultrachat_data(ds_split: Dataset):
    all_convos = []
    
    for ex in tqdm(ds_split, desc="Processing conversations"):
        messages = ex["messages"]
        
        # Reverse content and role names
        reversed_messages = []
        for msg in messages:
            role = USER_ROLE_NAME if msg["role"] == "user" else ASSISTANT_ROLE_NAME
            content = msg["content"].strip()[::-1]
            reversed_messages.append({"role": role, "content": content})
        
        # Create sliding windows with overlap
        windows = create_sliding_windows(reversed_messages, tokenizer, context_length)
        all_convos.extend(windows)
    
    return all_convos

def create_sliding_windows(messages, tokenizer, max_len):
    windows = []
    start_idx = 0
    
    while start_idx < len(messages):
        window = []
        i = start_idx
        
        # Build window by adding complete user-assistant pairs
        while i < len(messages):
            # Get next pair (user + optional assistant)
            pair = [messages[i]]
            if i + 1 < len(messages) and messages[i + 1]["role"] == ASSISTANT_ROLE_NAME:
                pair.append(messages[i + 1])
            
            # Test if adding this pair fits
            test_window = window + pair
            test_text = tokenizer.apply_chat_template(test_window, tokenize=False, add_generation_prompt=False)
            test_tokens = len(tokenizer.encode(test_text, truncation=False))
            
            if test_tokens > max_len:
                if not window:  # First pair is too big, skip it
                    # print(f"Skipping oversized pair with {test_tokens} tokens")
                    i += len(pair)  # Skip this pair
                    start_idx = i  # Continue from next position
                    break
                else:  # Window is ready
                    break
            
            window.extend(pair)
            i += len(pair)
        
        if window and len(window) >= 2:
            windows.append(window)
            
            # Simple overlap: start next window halfway through current window
            if len(window) >= 4:  # At least 2 pairs
                overlap_pairs = len(window) // 2  # Take half the pairs for overlap
                start_idx = start_idx + len(window) - overlap_pairs
            else:
                start_idx = i
        else:
            # If no valid window was created, ensure we advance
            if start_idx == i:
                start_idx += 1
            else:
                start_idx = i
    
    return windows

def filter_and_prepare_conversations(convos, tokenizer, max_len):
    filtered_convos = []
    for convo in tqdm(convos, desc="Filtering conversations"):
        if not convo:
            continue
        
        prompt_text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        tokenized_len = len(tokenizer.encode(prompt_text, truncation=False))
        
        if 0 < tokenized_len <= max_len:
            filtered_convos.append(convo)
    
    return {"conversations": filtered_convos}

def formatting_func(example):
    conversation = example["conversations"]
    
    prompt_text = tokenizer.apply_chat_template(conversation, tokenize=False, add_generation_prompt=False)
    
    tokenized_inputs = tokenizer(
        prompt_text,
        truncation=True,
        max_length=context_length,
        return_attention_mask=True,
        padding="max_length",
    )
    input_ids = tokenized_inputs["input_ids"]
    labels = [-100] * len(input_ids)
    
    # Use <im_start> and <im_end> tokens
    im_start_token_id = 52000
    im_end_token_id = 52001
    
    current_token_idx = 0
    for turn_idx, turn in enumerate(conversation):
        role = turn["role"]
        
        try:
            start_of_turn_bos_idx = input_ids.index(im_start_token_id, current_token_idx)
        except ValueError:
            break
        
        search_for_eos_from = start_of_turn_bos_idx + 1
        end_of_turn_eos_idx = -1
        
        for k_eos in range(search_for_eos_from, len(input_ids)):
            if input_ids[k_eos] == im_end_token_id:
                end_of_turn_eos_idx = k_eos
                break
        
        if end_of_turn_eos_idx == -1:
            break
        
        role_and_newline_text = f"{role}\n"
        role_and_newline_tokens = tokenizer.encode(role_and_newline_text, add_special_tokens=False)
        
        start_of_content_idx = start_of_turn_bos_idx + 1 + len(role_and_newline_tokens)
        
        # Mask all assistant responses
        if role == ASSISTANT_ROLE_NAME:
            for k_label in range(start_of_content_idx, end_of_turn_eos_idx + 1):
                if 0 <= k_label < len(labels):
                    labels[k_label] = input_ids[k_label]
        
        current_token_idx = end_of_turn_eos_idx + 1
    
    return {
        "input_ids": input_ids,
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": labels,
    }

# Process datasets
processed = {
    "train": process_ultrachat_data(raw_dataset[0]),
    "valid": process_ultrachat_data(raw_dataset[1]),
}

# Filter conversations
filtered_convos = {
    "train": filter_and_prepare_conversations(processed["train"], tokenizer, context_length),
    "valid": filter_and_prepare_conversations(processed["valid"], tokenizer, context_length),
}

print(f"Train conversations: {len(filtered_convos['train']['conversations'])}")
print(f"Valid conversations: {len(filtered_convos['valid']['conversations'])}")

# Tokenize and save
for split in ["train", "valid"]:
    tokenized_split = (
        Dataset.from_dict(filtered_convos[split])
        .map(formatting_func, remove_columns=["conversations"])
        .select_columns(["input_ids", "attention_mask", "labels"])
    )
    tokenized_split.save_to_disk(f"{DATA_DIR}/{dataset_name}/tokenized_{context_length}_{split}")

print("Tokenization complete!")